# NFL spread prediction

Train: 2012–2022, validation: 2023–2024, test: 2025.

Model is Ridge (alpha=1.0) on 4 features, no tuning. We already looked at 2025 in an earlier run, so it's not a truly untouched test set.

Data: nflverse `games.csv` (downloaded Oct 27, 2026). Closing spreads are only used for scoring, not as a feature.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from scipy.stats import binomtest

# regular season games 2012-2025 that have a final score
games = pd.read_csv('games.csv')
games = games[(games.game_type == 'REG') & games.season.between(2012, 2025)].copy()
games = games.dropna(subset=['home_score', 'away_score'])
# sort by date so we only use past games when building features
games = games.sort_values(['gameday', 'gametime', 'game_id'])
print('Completed regular-season games:', len(games))
print(games.groupby('season').size().to_string())

## Features
For each team, average points scored and allowed over its last 5 games (same season only). Skip a game if either team hasn't played yet that season.

Features (home minus away):
- `scored_diff`, `allowed_diff`
- `rest_diff`
- `home_field` (1 = home, 0 = neutral site)

In [ ]:
# history[(season, team)] = list of (points scored, points allowed) so far
history = {}
rows = []
skipped = 0
for _, game in games.iterrows():
    home_key = (game.season, game.home_team)
    away_key = (game.season, game.away_team)
    home = history.get(home_key, [])
    away = history.get(away_key, [])
    # need at least 1 previous game for both teams
    if len(home) > 0 and len(away) > 0:
        # average of last 5 games
        h = np.mean(home[-5:], axis=0)
        a = np.mean(away[-5:], axis=0)
        rows.append({
            'game_id': game.game_id, 'season': game.season, 'week': game.week,
            'home_team': game.home_team, 'away_team': game.away_team,
            'scored_diff': h[0] - a[0], 'allowed_diff': h[1] - a[1],
            'rest_diff': game.home_rest - game.away_rest,
            'home_field': int(game.location == 'Home'),
            'location': game.location,
            'margin': game.home_score - game.away_score,
            'spread': game.spread_line})
    else:
        skipped = skipped + 1
    # add this game's result after building features so there's no leakage
    history.setdefault(home_key, []).append((game.home_score, game.away_score))
    history.setdefault(away_key, []).append((game.away_score, game.home_score))
features = pd.DataFrame(rows)
columns = ['scored_diff', 'allowed_diff', 'rest_diff', 'home_field']
features = features.dropna(subset=columns)

# split by season
train = features[features.season.between(2012, 2022)].copy()
valid = features[features.season.between(2023, 2024)].copy()
historical = features[features.season == 2025].copy()
assert train.season.max() < valid.season.min()
assert valid.season.max() < historical.season.min()

# fit only on training data, predict home margin
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model.fit(train[columns], train.margin)
for name, frame in [('Training', train), ('Validation', valid), ('Historical evaluation', historical)]:
    print(name, ':', len(frame), 'games; locations:', frame.location.value_counts().to_dict())
print('Games skipped for insufficient history:', skipped)
valid['prediction'] = model.predict(valid[columns])
historical['prediction'] = model.predict(historical[columns])
features.to_csv('pregame_features.csv', index=False)


## Evaluation
Home covers if margin > spread (positive spread = home favored). Model picks home if prediction > spread. Baseline is always picking the favorite.

Dropped: pushes, pick'em games, missing spreads, model exactly equal to spread.

Each method gets accuracy with a 95% binomial CI and a test vs 50%. Paired test compares only games where the two methods disagree on correctness.

In [ ]:
def evaluate(frame, label):
    eligible = frame.dropna(subset=['spread']).copy()
    print('\n' + label)
    print('Margin MAE:', round(mean_absolute_error(frame.margin, frame.prediction), 3))
    # how far off the spread itself is, for comparison
    print('Spread-margin MAE:', round(mean_absolute_error(eligible.margin, eligible.spread), 3))
    print('Missing spreads:', int(frame.spread.isna().sum()))
    print('Pushes:', int((eligible.margin == eligible.spread).sum()))
    print("Pick'em games:", int((eligible.spread == 0).sum()))
    print('Exact model ties:', int((eligible.prediction == eligible.spread).sum()))
    # drop pushes, pick'ems, and ties
    scored = eligible[(eligible.margin != eligible.spread) & (eligible.spread != 0) & (eligible.prediction != eligible.spread)].copy()
    scored['home_cover'] = scored.margin > scored.spread
    scored['model_home'] = scored.prediction > scored.spread
    scored['favorite_home'] = scored.spread > 0
    scored['model_correct'] = scored.model_home == scored.home_cover
    scored['favorite_correct'] = scored.favorite_home == scored.home_cover

    # accuracy overall, per season, and early vs late weeks
    results = []
    groups = [('All', scored)]
    for season in sorted(scored.season.unique()):
        groups.append((str(season), scored[scored.season == season]))
    groups.extend([('Weeks 2-5', scored[scored.week <= 5]), ('Weeks 6-18', scored[scored.week >= 6])])
    for group, subset in groups:
        for method in ['model', 'favorite']:
            n = len(subset)
            if n == 0:
                continue
            k = int(subset[method + '_correct'].sum())
            test = binomtest(k, n, p=0.5)
            ci = test.proportion_ci()
            results.append({'split': label, 'group': group, 'method': method, 'n': n, 'correct': k, 'accuracy': k/n, 'ci_low': ci.low, 'ci_high': ci.high, 'p_vs_half': test.pvalue})
    results = pd.DataFrame(results)
    print(results.round(4).to_string(index=False))

    # paired test: only games where one method is right and the other is wrong
    wins = int((scored.model_correct & ~scored.favorite_correct).sum())
    losses = int((~scored.model_correct & scored.favorite_correct).sum())
    print('Model-only correct:', wins, 'Favorite-only correct:', losses)
    if wins + losses > 0:
        print('Paired exact p-value:', binomtest(wins, wins + losses).pvalue)
    return scored, results

validation_predictions, validation_results = evaluate(valid, 'Validation 2023-2024')
validation_predictions.to_csv('validation_predictions.csv', index=False)
validation_results.to_csv('validation_results.csv', index=False)


## 2025 results
Already seen in an earlier run, so don't use this to pick settings. Use validation for that.

In [4]:
historical_predictions, historical_results = evaluate(historical, 'Historical 2025 (previously inspected)')
historical_predictions.to_csv('historical_2025_predictions.csv', index=False)
historical_results.to_csv('historical_2025_results.csv', index=False)



Historical 2025 (previously inspected)
Margin MAE: 10.561
Spread-margin MAE: 9.959
Missing spreads: 0
Pushes: 1
Pick'em games: 0
Exact model ties: 0
                                 split      group   method   n  correct  accuracy  ci_low  ci_high  p_vs_half
Historical 2025 (previously inspected)        All    model 255      133    0.5216  0.4584   0.5843     0.5313
Historical 2025 (previously inspected)        All favorite 255      122    0.4784  0.4157   0.5416     0.5313
Historical 2025 (previously inspected)       2025    model 255      133    0.5216  0.4584   0.5843     0.5313
Historical 2025 (previously inspected)       2025 favorite 255      122    0.4784  0.4157   0.5416     0.5313
Historical 2025 (previously inspected)  Weeks 2-5    model  62       28    0.4516  0.3248   0.5832     0.5258
Historical 2025 (previously inspected)  Weeks 2-5 favorite  62       32    0.5161  0.3856   0.6450     0.8991
Historical 2025 (previously inspected) Weeks 6-18    model 193      105    0.544

In [6]:
%pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.5 MB 5.2 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/9.5 MB 5.2 MB/s eta 0:00:02
   ------------- -------------------------- 3.1/9.5 MB 5.2 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.5 MB 5.2 MB/s eta 0:00:02
   ---------------------- ----------------- 5.2/9.5 MB 5.2 MB/s eta 0:00:01
   -------------------------- ------------- 6.3/9.5 MB 5.2 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.5 MB 5.2 MB/s eta 0:00:01
   ------------------------------------ --- 8.7/9.5 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 5.1 MB/s  0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.5 MB 


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import matplotlib.pyplot as plt

validation = pd.read_csv("validation_results.csv")
historical = pd.read_csv("historical_2025_results.csv")

results = pd.concat([validation, historical], ignore_index=True)
seasons = ["2023", "2024", "2025"]
x = np.arange(3)

fig, ax = plt.subplots(figsize=(6.5, 3.5))

# side by side bars for model vs favorite, with CI error bars
for method, label, offset, color in [
    ("model", "Ridge regression", -0.18, "steelblue"),
    ("favorite", "Favorite-only", 0.18, "darkorange")
]:
    # only keep the per-season rows
    rows = results[
        (results["method"] == method)
        & results["group"].astype(str).isin(seasons)
    ].copy()

    rows["group"] = rows["group"].astype(str)
    rows = rows.set_index("group").loc[seasons]

    accuracy = rows["accuracy"].to_numpy() * 100
    lower = rows["ci_low"].to_numpy() * 100
    upper = rows["ci_high"].to_numpy() * 100

    ax.bar(
        x + offset, accuracy, width=0.36,
        label=label, color=color,
        yerr=[accuracy - lower, upper - accuracy],
        capsize=3
    )

# 50% = coin flip
ax.axhline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([
    "2023\nValidation",
    "2024\nValidation",
    "2025\nHistorical evaluation"
])
ax.set_ylabel("Correct spread picks (%)")
ax.set_ylim(0, 70)
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.savefig("accuracy_by_season.png", dpi=300, bbox_inches="tight")
fig.savefig("accuracy_by_season.pdf", bbox_inches="tight")
plt.show()

## Sources
- https://github.com/nflverse/nfldata
- https://nflreadr.nflverse.com/articles/dictionary_schedules.html
- https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html